# El Niño & Indian Air Pollution — SFT + QLoRA

**Goal:** fine-tune a small instruction-following language model to answer climate and air-pollution questions in a neutral, evidence-aware way.

### What this notebook does
1. Loads a public climate Q&A dataset from Hugging Face.
2. Adds a small, curated set of Q&A examples based on our selected El Niño article.
3. Loads `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit mode.
4. Applies **QLoRA**: the base model stays quantized/frozen while small LoRA adapter weights are trained.
5. Compares sample answers before and after fine-tuning, evaluates loss, and saves the adapter.

**Scope note:** this is a short academic prototype, not a production climate-information service. The public dataset covers climate change broadly, while the article examples provide the India/ENSO/pollution focus. The model should not be treated as an official forecast or policy authority.

## 0. Before running

In Google Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Then run every cell from top to bottom.

The training subset is deliberately small to fit a roughly three-hour assignment session. Internet access is needed to download the model and dataset.

In [ ]:
import os, json, random, time, shutil, platform
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Switch Colab runtime to a GPU before training.")

## 1. Install compatible libraries

This notebook uses Transformers + PEFT + bitsandbytes. `Trainer` performs supervised fine-tuning (SFT); PEFT supplies LoRA adapters, and bitsandbytes supplies 4-bit quantization.

In [ ]:
%pip -q install "transformers==4.51.3" "peft==0.15.2" "bitsandbytes==0.45.5" "accelerate==1.6.0" "datasets==3.5.0"

## 2. Prepare the training data

**Public dataset:** [`pierre-pessarossi/climate-question-answers`](https://huggingface.co/datasets/pierre-pessarossi/climate-question-answers), an English climate Q&A dataset with `instruction` and `answer` columns (MIT license).

We use a small sample of its **training split**, then add hand-curated examples based on the selected article. The article examples include the study's reported figures and its key caveat: the results show associations for particular winters, not proof that El Niño always worsens pollution.

In [ ]:
from datasets import load_dataset, Dataset

HF_DATASET_ID = "pierre-pessarossi/climate-question-answers"
HF_SAMPLE_SIZE = 120  # small enough for a quick prototype

article_qa = [
    ("What did the study find about India's winter PM2.5 patterns?",
     "The study reported that winter PM2.5 patterns differed between the La Niña winter of 2022–23 and the El Niño winter of 2023–24. In 2022–23, Mumbai PM2.5 was about 45% above the 2019–22 average. In 2023–24, Jaipur was about 50% higher and Delhi more than 10% higher, while parts of peninsular India were cleaner."),
    ("Does El Niño always make air pollution worse in India?",
     "No. The study compares particular winters and reports associations; it does not show that El Niño always worsens pollution. The authors caution against treating the results as proof of causation or extrapolating them to other years."),
    ("What is PM2.5?",
     "PM2.5 is particulate matter with a diameter of 2.5 micrometres or smaller. These fine particles can enter deep into the respiratory system. Air-quality reports commonly express PM2.5 concentration in micrograms per cubic metre."),
    ("What is ENSO?",
     "ENSO means El Niño–Southern Oscillation, a natural climate pattern involving warm El Niño and cool La Niña phases in the tropical Pacific. It can influence weather patterns in other regions, including winds and rainfall over India."),
    ("Why can pollution rise even if emissions have not changed much?",
     "Weather affects how pollutants disperse or accumulate. Weak winds and stagnant air can allow PM2.5 to build up near the ground, while stronger winds and some rainfall conditions can help disperse or remove pollution. Emissions still matter."),
    ("What happened in Mumbai during winter 2022–23?",
     "The article reports that Mumbai's PM2.5 was about 45% above the 2019–22 average during the La Niña winter of 2022–23. This is a reported comparison for that period, not a general rule for every La Niña year."),
    ("Which cities had higher PM2.5 in winter 2023–24 in the article?",
     "The article reports that Jaipur's PM2.5 was about 50% higher and Delhi's was more than 10% higher than the 2019–22 average. Parts of peninsular India were cleaner during that winter."),
    ("What does the study say about the size of year-to-year PM2.5 changes?",
     "It reports regional year-to-year changes ranging from roughly a 40% decline to a 70% increase compared with the 2019–22 average, without a major change in emissions or pollution-control policy. These figures describe the studied period and should not be generalized automatically."),
    ("Why is regional coordination important for air pollution policy?",
     "Polluted air can travel across city and state boundaries, so a city's air quality may be affected by emissions and air masses from elsewhere. Coordinated airshed-level planning can complement local emission controls."),
    ("Does this study prove weather caused every observed pollution change?",
     "No. The authors report relationships between weather patterns and PM2.5, but correlation alone does not establish direct causation. The findings should be interpreted cautiously and checked against more data."),
    ("What is an airshed?",
     "An airshed is a geographic area where air pollution moves and mixes across administrative boundaries. Managing air quality at airshed level can help neighbouring jurisdictions coordinate emission reductions."),
    ("What policy could help prepare for weather-related pollution episodes?",
     "One option is to combine seasonal climate indicators and weather forecasts with air-quality planning. Forecasts are uncertain, so they should support—not replace—continuous emission reduction and public-health measures."),
    ("Why should El Niño information not replace emission controls?",
     "El Niño and other weather patterns can influence where pollution accumulates, but they do not eliminate the role of emissions. Long-term emission reduction remains essential even when weather temporarily improves air quality."),
    ("What agricultural issue does the related climate-zone research raise?",
     "The related research described a shift in climate zones, including the hot semi-arid zone extending eastward into Uttar Pradesh. This suggests that local agricultural advisories and crop-calendar planning may need to account for changing climate conditions."),
    ("What is a limitation of using only one or two winters to make policy?",
     "A small number of winters cannot establish a universal relationship. Climate and pollution vary from year to year, so policy should use longer records, updated observations, and uncertainty-aware forecasts."),
    ("How should a public-facing climate assistant respond when evidence is incomplete?",
     "It should say what the evidence supports, distinguish observations from causal claims, acknowledge uncertainty, and avoid inventing statistics. For operational decisions, it should point users to current official data and expert guidance."),
    ("What is one evidence-based air-quality policy recommendation from this topic?",
     "Develop seasonal, climate-aware air-quality forecasts and coordinate action across airsheds. Such forecasts should be evaluated against observations and used alongside—not instead of—long-term emission controls."),
    ("What should citizens understand about a cleaner winter?",
     "Cleaner air in a particular winter does not necessarily prove that emissions have fallen. Weather can affect pollutant dispersion, so air-quality trends should be assessed alongside emissions data and longer-term observations."),
    ("What does a percentage increase relative to an average mean?",
     "It compares a measured value with a reference average. The interpretation depends on the chosen reference period and data quality, so the comparison period should always be stated."),
    ("What is the difference between an observation and a policy recommendation?",
     "An observation describes what the data or study reports. A policy recommendation proposes an action based on evidence, expected benefits, costs, uncertainty, and practical constraints.")
]

SYSTEM_PROMPT = (
    "You are AirClimate Assistant, a plain-English educational assistant focused on climate, "
    "ENSO, PM2.5, air pollution, and evidence-based public policy. Be neutral and concise. "
    "Distinguish correlation from causation. Never claim El Niño always worsens pollution. "
    "Do not invent statistics or claim to provide live forecasts. If evidence is insufficient, say so."
)

# Load the public climate Q&A training split. A small article-only fallback is kept
# so the notebook can still demonstrate the pipeline if the Hub is temporarily unavailable.
hf_rows = []
try:
    public_ds = load_dataset(HF_DATASET_ID, split="train")
    public_ds = public_ds.shuffle(seed=SEED).select(range(min(HF_SAMPLE_SIZE, len(public_ds))))
    for row in public_ds:
        q = str(row.get("instruction", "")).strip()
        a = str(row.get("answer", "")).strip()
        if q and a and q.lower() != "nan" and a.lower() != "nan":
            hf_rows.append((q, a))
    print(f"Loaded {len(hf_rows)} Q&A pairs from {HF_DATASET_ID}.")
except Exception as e:
    print("Could not load the public Hugging Face dataset:", repr(e))
    print("Continuing with the curated article examples only. If this happens, retry after checking internet access.")

# Hold back four article examples for a small qualitative check.
article_train = article_qa[:-4]
article_eval = article_qa[-4:]

# Split the public sample into train and validation portions.
split_point = max(1, int(0.9 * len(hf_rows)))
hf_train = hf_rows[:split_point]
hf_eval = hf_rows[split_point:]

def make_messages(q, a):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a}
    ]

train_pairs = hf_train + article_train
eval_pairs = hf_eval + article_eval

random.Random(SEED).shuffle(train_pairs)
random.Random(SEED).shuffle(eval_pairs)

train_messages = [make_messages(q, a) for q, a in train_pairs]
eval_messages = [make_messages(q, a) for q, a in eval_pairs]

print("Article-specific training examples:", len(article_train))
print("Public climate Q&A training examples:", len(hf_train))
print("Total training examples:", len(train_messages))
print("Evaluation examples:", len(eval_messages))
print("\nExample article Q&A:")
print("Q:", article_train[0][0])
print("A:", article_train[0][1])

## 3. Load the base model in 4-bit mode

QLoRA reduces GPU memory use by loading the base model in 4-bit NF4 quantization. The base weights are kept frozen; only small LoRA adapter weights will be trained.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = 512

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this 4-bit QLoRA notebook. In Colab select Runtime > Change runtime type > T4 GPU.")

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Tokenize the chat examples

Each example is formatted using the model's own chat template. We keep the sequence length modest for the time and GPU budget.

In [ ]:
def to_chat_text(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

train_texts = [to_chat_text(m) for m in train_messages]
eval_texts = [to_chat_text(m) for m in eval_messages]

train_raw = Dataset.from_dict({"text": train_texts})
eval_raw = Dataset.from_dict({"text": eval_texts})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

train_tokenized = train_raw.map(tokenize_batch, batched=True, remove_columns=["text"])
eval_tokenized = eval_raw.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Tokenized train rows:", len(train_tokenized))
print("Tokenized evaluation rows:", len(eval_tokenized))

## 5. Compare a few answers before training

These are qualitative examples, not a formal benchmark. We'll ask the same questions after training.

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_questions = [
    "Does El Niño always make India's air pollution worse?",
    "Why can pollution rise even if emissions have not changed much?",
    "What should policymakers do about weather-related pollution?"
]

before_answers = {}
for question in test_questions:
    answer = generate_answer(model, tokenizer, question)
    before_answers[question] = answer
    print("\nQUESTION:", question)
    print("BASE MODEL:", answer)

## 6. Run supervised fine-tuning (SFT)

The trainer optimizes the next-token prediction objective on the Q&A examples. QLoRA keeps the base model quantized and trains only the small adapter weights. This is a compact demonstration; lower training loss does not by itself prove factual accuracy.

In [ ]:
OUTPUT_DIR = "/content/airclimate_qlora_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir="/content/airclimate_training_logs",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=not use_bf16,
    bf16=use_bf16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    group_by_length=True,
    remove_unused_columns=False,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized if len(eval_tokenized) else None,
    data_collator=data_collator
)

start_time = time.time()
train_result = trainer.train()
train_minutes = round((time.time() - start_time) / 60, 2)

print("Training finished in", train_minutes, "minutes.")
print("Training metrics:", train_result.metrics)

## 7. Evaluate and compare after fine-tuning

We report evaluation loss and compare the same example questions. Treat the answers as qualitative evidence only; review them for correctness and uncertainty.

In [ ]:
eval_metrics = {}
try:
    eval_metrics = trainer.evaluate()
    print("Evaluation metrics:", eval_metrics)
except Exception as e:
    print("Evaluation could not be completed:", repr(e))

model.config.use_cache = True
after_answers = {}
for question in test_questions:
    answer = generate_answer(trainer.model, tokenizer, question)
    after_answers[question] = answer
    print("\nQUESTION:", question)
    print("BEFORE:", before_answers[question])
    print("AFTER :", answer)

## 8. Save the adapter and run summary

The adapter is much smaller than a full model checkpoint. The notebook saves the LoRA adapter and tokenizer, then creates a ZIP file for download.

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())

summary = {
    "project": "El Nino and Indian Air Pollution - SFT + QLoRA",
    "base_model": MODEL_ID,
    "dataset": HF_DATASET_ID if hf_rows else "curated article Q&A fallback",
    "public_hf_examples_used": len(hf_rows),
    "article_examples_total": len(article_qa),
    "training_examples": len(train_messages),
    "evaluation_examples": len(eval_messages),
    "epochs": 2,
    "max_sequence_length": MAX_LENGTH,
    "quantization": "4-bit NF4 with double quantization",
    "lora_rank": 8,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "trainable_parameters": int(trainable_params),
    "total_parameters_in_wrapped_model": int(total_params),
    "train_minutes": train_minutes,
    "evaluation_metrics": {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))}
}
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

zip_base = "/content/airclimate_qlora_adapter"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print("\nAdapter ZIP created at:", zip_base + ".zip")

In [ ]:
from google.colab import files
files.download("/content/airclimate_qlora_adapter.zip")

## 9. What to report in the assignment

Record:
- Base model: `Qwen/Qwen2.5-1.5B-Instruct`
- Method: supervised fine-tuning with QLoRA (4-bit NF4 + LoRA rank 8)
- Dataset: Hugging Face climate Q&A training subset + curated Q&A derived from the selected El Niño article
- GPU, training time, training/evaluation loss, and before/after examples from `run_summary.json` and notebook output

**Limitations:** the article-specific set is small; the public climate dataset is broader than Indian air pollution; answers require human fact-checking; this is not a live forecasting model. Keep the article's uncertainty caveat in the report: the study reports associations for specific winters and does not prove that El Niño always worsens pollution.